# EP flux and dynamical indices

Raw inputs are read from `PAPER1_ARCHIVE_ROOT`; preprocessing products are read from `PAPER1_PREPROCESSED_ROOT`; new diagnostics are written to `PAPER1_DERIVED_ROOT` (default: repository-local `work/`). Source files are never modified.


## One EP-flux implementation with source-specific exact grids

Inputs: staged U/V/T with identical date/time/lat/lon/pressure coordinates. Outputs: coordinate-checking EP writers. Method: MERRA-2 and free-running WACCM use their common exact 18-level hPa grid; restart cases use the exact 23-level grid. All call `ComputeEPfluxDiv` with `do_ubar=True`, `w=None`, `wave=-1`, natural-calendar-month N2, upward `-ep2`, and a 40--80N cosine mean.


In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

def discover_diagnostic_directory():
    candidates = (
        Path.cwd(), Path.cwd() / "analysis",
        Path.cwd() / "Paper1" / "analysis",
    )
    for candidate in candidates:
        if (candidate / "lib" / "workflow_io.py").is_file():
            return candidate.resolve()
    raise FileNotFoundError(
        "Cannot locate Paper1/analysis/lib from the current working directory"
    )

NOTEBOOK_DIR = discover_diagnostic_directory()
LIB = NOTEBOOK_DIR / "lib"
if str(LIB) not in sys.path:
    sys.path.insert(0, str(LIB))

from workflow_io import (
    PRODUCT_VERSION, archive_root, derived_root, preprocessed_root, product_path,
    write_csv_atomic, write_netcdf_atomic,
)

ARCHIVE_ROOT = archive_root()
PREPROCESSED_ROOT = preprocessed_root()
DERIVED_ROOT = derived_root()
MARCH_HINDCAST_ROOT = Path(os.environ.get(
    "PAPER1_MARCH_HINDCAST_SOURCE",
    ""
    "",
))
OVERWRITE = os.environ.get("PAPER1_OVERWRITE_STAGING", "0") == "1"
print("read-only archive root:", ARCHIVE_ROOT)
print("preprocessed staging input root:", PREPROCESSED_ROOT)
print("read-only March hindcast root:", MARCH_HINDCAST_ROOT)
print("staging output root:", DERIVED_ROOT)
print("diagnostic notebook directory:", NOTEBOOK_DIR)

import hashlib

from epflux_jucker import ComputeEPfluxDiv, LICENSE as EP_LICENSE, SOURCE_URL as EP_SOURCE_URL
from paper1_diagnostics import (
    EP_FREE_PLEV_HPA, EP_RESTART_PLEV_HPA, PLEV_HPA,
    compute_epflux_monthly, date_int, hybrid_mid_pressure,
    log_pressure_interpolate, parse_member, parse_year, select_latitude,
    select_pressure_levels,
)

EP_IMPLEMENTATION = LIB / "epflux_jucker.py"
EP_SHA256 = hashlib.sha256(EP_IMPLEMENTATION.read_bytes()).hexdigest()

def waccm_uvt(root, year):
    files = {}
    for variable in ("U", "V", "T"):
        matches = [
            path for path in (root / "interpolated" / variable).glob(f"*.{variable}.nc")
            if parse_year(path) == year
        ]
        if len(matches) != 1:
            raise RuntimeError(f"{root.name} {year:04d} {variable}: expected one input; found {len(matches)}")
        files[variable] = matches[0]
    return pressure_uvt([files["U"], files["V"], files["T"]], EP_FREE_PLEV_HPA)

def pressure_uvt(paths, target_levels):
    """Read staged U/V/T only after exact calendar/grid identity checks."""

    datasets = [xr.open_dataset(path, decode_times=True, chunks={"time": 8}) for path in paths]
    try:
        if len(datasets) == 1:
            sources = {variable: datasets[0] for variable in ("U", "V", "T")}
        elif len(datasets) == 3:
            sources = dict(zip(("U", "V", "T"), datasets))
        else:
            raise ValueError("pressure_uvt expects one combined file or three U/V/T files")
        for variable, source in sources.items():
            if variable not in source:
                raise KeyError(f"{paths}: missing {variable}")
        reference_source = sources["U"]
        reference_field = reference_source["U"]
        reference_dates = date_int(reference_source)
        reference_time = np.asarray(reference_source.time.values)
        reference_lat = np.asarray(reference_field.lat.values)
        reference_lon = np.asarray(reference_field.lon.values)
        reference_pname = "plev" if "plev" in reference_field.coords else "lev"
        reference_plev = np.asarray(reference_field[reference_pname].values, dtype=float)
        if np.nanmax(reference_plev) > 2000.0:
            reference_plev = reference_plev / 100.0
        for variable in ("V", "T"):
            source = sources[variable]
            field = source[variable]
            pname = "plev" if "plev" in field.coords else "lev"
            plev = np.asarray(field[pname].values, dtype=float)
            if np.nanmax(plev) > 2000.0:
                plev = plev / 100.0
            checks = {
                "integer date": (reference_dates, date_int(source)),
                "time": (reference_time, np.asarray(source.time.values)),
                "latitude": (reference_lat, np.asarray(field.lat.values)),
                "longitude": (reference_lon, np.asarray(field.lon.values)),
                "pressure (hPa)": (reference_plev, plev),
            }
            for coordinate, (expected, actual) in checks.items():
                if not np.array_equal(expected, actual):
                    raise RuntimeError(f"U/{variable} {coordinate} coordinates differ")
        fields = []
        for variable in ("U", "V", "T"):
            field = select_pressure_levels(sources[variable][variable], target_levels)
            field = field.transpose("time", "plev", "lat", "lon").load()
            fields.append(field)
    finally:
        for dataset in datasets:
            dataset.close()
    return (*fields, reference_dates)

def dynamics_from_uvt(u, t, ep):
    zonal_u = u.mean("lon", skipna=True)
    u60n10 = zonal_u.interp(lat=60.0).sel(plev=10.0).rename("u60n10")
    zonal_t = select_latitude(t.mean("lon", skipna=True), 60.0, 90.0)
    tmin50 = zonal_t.sel(plev=50.0).min("lat", skipna=True).rename("tmin50")
    return xr.Dataset({
        "u60n10": u60n10,
        "tmin50": tmin50,
        "ep100_upward_40_80n": ep.ep100_upward_40_80n,
    }).assign_coords(date=ep.date)

def write_ep(dataset, name, *, expected_plev, members=False):
    machine_method = {
        "natural_month_n2": "True", "do_ubar": "True",
        "w_argument": "None", "wave": "-1",
    }
    for attribute, expected in machine_method.items():
        if str(dataset.attrs.get(attribute)) != expected:
            raise RuntimeError(
                f"{name}: EP machine attr {attribute}={dataset.attrs.get(attribute)!r}; "
                f"expected {expected!r}"
            )
    actual_plev = np.asarray(dataset.plev.values, dtype=float)
    expected_plev = np.asarray(expected_plev, dtype=float)
    if not np.array_equal(actual_plev, expected_plev):
        raise RuntimeError(f"{name}: EP pressure grid differs from its declared hPa grid")
    if any(level not in actual_plev for level in (10.0, 50.0, 100.0)):
        raise RuntimeError(f"{name}: EP pressure grid lacks 10, 50, or 100 hPa")
    dataset.plev.attrs.update(units="hPa", positive="down")
    dataset.attrs.update(
        epflux_implementation="vendored Methods-only epflux_jucker.py",
        epflux_source_url=EP_SOURCE_URL, epflux_license=EP_LICENSE,
        epflux_source_sha256=EP_SHA256,
        pressure_grid_hpa=",".join(f"{level:g}" for level in expected_plev),
    )
    prefix = ("member",) if members else ()
    exact_sizes = {"plev": len(expected_plev)}
    if members:
        exact_sizes["member"] = 30
    write_netcdf_atomic(
        dataset, product_path("epflux", name),
        required_vars={
            "ep1": prefix + ("time", "plev", "lat"),
            "ep2_raw": prefix + ("time", "plev", "lat"),
            "fz_upward": prefix + ("time", "plev", "lat"),
            "fz_upward_40_80n": prefix + ("time", "plev"),
            "ep100_upward_40_80n": prefix + ("time",),
        }, required_coords=("date", "plev", "lat") + (("member",) if members else ()),
        exact_sizes=exact_sizes,
        required_attrs={
            "epflux_source_sha256": EP_SHA256,
            "natural_month_n2": "True", "do_ubar": "True",
            "w_argument": "None", "wave": "-1",
        }, overwrite=OVERWRITE,
    )

def write_dynamics(dataset, name, *, members=False):
    prefix = ("member",) if members else ()
    write_netcdf_atomic(
        dataset, product_path("dynamics", name),
        required_vars={
            "u60n10": prefix + ("time",), "tmin50": prefix + ("time",),
            "ep100_upward_40_80n": prefix + ("time",),
        }, required_coords=("date",) + (("member",) if members else ()),
        exact_sizes=({"member": 30} if members else None), overwrite=OVERWRITE,
    )


## Ranked WACCM events plus audited predecessor EP years

Inputs: the strict 207+23 ranking, the 01-preprocessing 209/23 pressure-year manifests, and staged pressure-level U/V/T. Outputs: full-latitude upward flux, 40--80N pressure profiles, and EP100 for exactly the audited source years. Method: ranked springs define membership; predecessor-only source years supply October--December to their ranked successor but never become independent climatology samples.


In [ ]:
manifest = pd.read_csv(product_path("ozone", "waccm_event_manifest.csv"))
for segment, root, output_name in (
    ("LONGRUN", PREPROCESSED_ROOT / "B2000WCN001002_timefixed", "waccm_longrun_epflux.nc"),
    ("BWCN", PREPROCESSED_ROOT / "BWCN", "waccm_bwcn_epflux.nc"),
):
    ranked_years = set(
        manifest.loc[manifest.source_segment.astype(str) == segment, "model_year"].astype(int)
    )
    expected_ranking = 207 if segment == "LONGRUN" else 23
    expected_pressure = 209 if segment == "LONGRUN" else 23
    selection_path = (
        PREPROCESSED_ROOT / "manifests" /
        f"waccm_{segment.lower()}_pressure_years.tsv"
    )
    selection = pd.read_csv(selection_path, sep="\t")
    required = {
        "segment", "pressure_year", "raw_available", "ranking_event_complete",
        "needed_as_predecessor", "pressure_source_required",
    }
    if required - set(selection):
        raise ValueError(f"{selection_path}: missing {sorted(required - set(selection))}")
    if set(selection.segment.astype(str)) != {segment}:
        raise RuntimeError(f"{selection_path}: segment column is not exactly {segment}")
    if selection.pressure_year.astype(int).duplicated().any():
        raise RuntimeError(f"{selection_path}: duplicate pressure years")
    audited_ranking = set(
        selection.loc[
            selection.ranking_event_complete.astype(int) == 1, "pressure_year"
        ].astype(int)
    )
    pressure_years = set(
        selection.loc[
            selection.pressure_source_required.astype(int) == 1, "pressure_year"
        ].astype(int)
    )
    predecessor_years = set(
        selection.loc[
            selection.needed_as_predecessor.astype(int) == 1, "pressure_year"
        ].astype(int)
    )
    if ranked_years != audited_ranking:
        raise RuntimeError(f"{segment}: ozone ranking and pressure-year manifest differ")
    if pressure_years != ranked_years | predecessor_years:
        raise RuntimeError(f"{segment}: pressure-source union is inconsistent")
    if len(ranked_years) != expected_ranking or len(pressure_years) != expected_pressure:
        raise RuntimeError(
            f"{segment}: ranking/pressure counts={(len(ranked_years), len(pressure_years))}; "
            f"expected={(expected_ranking, expected_pressure)}"
        )
    selected_rows = selection.loc[selection.pressure_source_required.astype(int) == 1]
    if not (selected_rows.raw_available.astype(int) == 1).all():
        raise RuntimeError(f"{segment}: required pressure source is not raw-available")
    # Figure 2 needs October--December and Figure 15 needs November--December
    # from the predecessor model year.  The audited pressure-year manifest is
    # authoritative; a predecessor never becomes an independent ranked event.
    available_by_variable = {
        variable: {
            parse_year(path)
            for path in (root / "interpolated" / variable).glob(f"*.{variable}.nc")
        }
        for variable in ("U", "V", "T")
    }
    complete_uvt_years = set.intersection(*available_by_variable.values())
    missing_pressure = sorted(pressure_years - complete_uvt_years)
    if missing_pressure:
        raise FileNotFoundError(
            f"{segment}: pressure-manifest EP years lack complete U/V/T: {missing_pressure[:5]}"
        )
    requested_padding = {year - 1 for year in ranked_years} - ranked_years
    available_padding = pressure_years - ranked_years
    missing_padding = sorted(requested_padding - pressure_years)
    years = sorted(pressure_years)
    chunk_paths = []
    for year in years:
        u, v, t, dates = waccm_uvt(root, year)
        dates = np.asarray(dates, dtype=int)
        if (
            dates.size != 365
            or np.unique(dates).size != 365
            or not np.all(dates // 10000 == year)
            or np.any(((dates // 100) % 100 == 2) & (dates % 100 == 29))
        ):
            raise RuntimeError(f"{segment} {year:04d}: expected one exact 365-day no-leap year")
        result = compute_epflux_monthly(u, v, t, dates, ComputeEPfluxDiv)
        result = result.assign_coords(model_year=("time", np.full(result.sizes["time"], year)))
        result.attrs.update(
            product_version=PRODUCT_VERSION, source_segment=segment, model_year=int(year),
            ranked_ozone_event="True" if year in ranked_years else "False (Oct--Dec predecessor padding only)",
            natural_month_n2="True", do_ubar="True", w_argument="None", wave="-1",
        )
        chunk_name = f"_chunks/{segment.lower()}/{year:04d}.nc"
        write_ep(result, chunk_name, expected_plev=EP_FREE_PLEV_HPA)
        chunk_paths.append(product_path("epflux", chunk_name))
        del u, v, t, result
    combined = xr.open_mfdataset(
        chunk_paths, combine="nested", concat_dim="time", decode_times=False,
        chunks={"time": 365},
    )
    try:
        combined.attrs.update(
            product_version=PRODUCT_VERSION, source_segment=segment,
            ranked_event_count=expected_ranking, ep_processed_year_count=len(years),
            pressure_year_manifest=str(selection_path),
            ep_padding_model_years=",".join(f"{year:04d}" for year in sorted(available_padding)),
            unavailable_ep_padding_model_years=",".join(f"{year:04d}" for year in missing_padding),
            natural_month_n2="True", do_ubar="True", w_argument="None", wave="-1",
        )
        write_ep(combined, output_name, expected_plev=EP_FREE_PLEV_HPA)
    finally:
        combined.close()


## Explicit BWCN year-0008 dynamics

Inputs: staged BWCN year-0008 U/T and canonical BWCN EP flux. Outputs: dynamics/waccm_bwcn_year0008.nc. Method: produce only U60N10, Tmin50, and reference EP100 from coordinate-selected year 0008.


In [ ]:
u, v, t, dates = waccm_uvt(PREPROCESSED_ROOT / "BWCN", 8)
with xr.open_dataset(product_path("epflux", "waccm_bwcn_epflux.nc"), decode_times=False) as source:
    selected = np.asarray(source.model_year.values, dtype=int) == 8
    reference_ep = source.isel(time=np.flatnonzero(selected)).load()
reference_dynamics = dynamics_from_uvt(u, t, reference_ep)
reference_dynamics.attrs.update(
    product_version=PRODUCT_VERSION, reference_event="BWCN year 0008",
    tmin_definition="minimum of zonal-mean T over 60--90N at 50 hPa",
    source_pressure_levels="staged exact 18-level free-running EP subset in hPa",
)
write_dynamics(reference_dynamics, "waccm_bwcn_year0008.nc")


## MERRA-2 1980--2025 EP flux

Inputs: staged MERRA-2 U/V/T at the audited 18 exact pressure levels. Outputs: epflux/merra2_1980_2025_epflux.nc. Method: all 46 years use the identical monthly-natural-calendar-N2/all-wave/no-omega implementation, with pressure stored in hPa.


In [ ]:
chunk_paths = []
for year in range(1980, 2026):
    paths = [PREPROCESSED_ROOT / "MERRA2_Processed" / variable / f"MERRA2.{variable}.{year}.nc" for variable in ("U", "V", "T")]
    missing = [str(path) for path in paths if not path.exists()]
    if missing:
        raise FileNotFoundError(f"MERRA-2 U/V/T missing for {year}: {missing}")
    u, v, t, dates = pressure_uvt(paths, EP_FREE_PLEV_HPA)
    result = compute_epflux_monthly(u, v, t, dates, ComputeEPfluxDiv)
    result.attrs.update(
        product_version=PRODUCT_VERSION, source_period=str(year),
        natural_month_n2="True", do_ubar="True", w_argument="None", wave="-1",
    )
    chunk_name = f"_chunks/merra2/{year:04d}.nc"
    write_ep(result, chunk_name, expected_plev=EP_FREE_PLEV_HPA)
    chunk_paths.append(product_path("epflux", chunk_name))
    del u, v, t, result
merra_ep = xr.open_mfdataset(
    chunk_paths, combine="nested", concat_dim="time", decode_times=False,
    chunks={"time": 365},
)
try:
    merra_ep.attrs.update(
        product_version=PRODUCT_VERSION, source_period="1980--2025",
        natural_month_n2="True", do_ubar="True", w_argument="None", wave="-1",
    )
    write_ep(merra_ep, "merra2_1980_2025_epflux.nc", expected_plev=EP_FREE_PLEV_HPA)
finally:
    merra_ep.close()


## Restart producer definitions

Inputs: staged CDO Jan/Feb pressure-level files and raw March hindcast pressure-level files. Outputs: a reusable coordinate-checking producer. Method: require 30 common member IDs and one common integer-date calendar.


In [ ]:
def hindcast_case(case, root, *, pressure_level, output_kind):
    if output_kind not in {"epflux", "dynamics"}:
        raise ValueError(output_kind)
    if pressure_level and (root / "interpolated").is_dir():
        variable_maps = {
            variable: {
                parse_member(path): path
                for path in (root / "interpolated" / variable).glob(f"*.{variable}.nc")
            }
            for variable in ("U", "V", "T")
        }
        common = sorted(set.intersection(*(set(mapping) for mapping in variable_maps.values())))
        grouped = [(member, [variable_maps[v][member] for v in ("U", "V", "T")]) for member in common]
    elif pressure_level:
        members = sorted(root.glob("*.nc"))
        grouped = [(path, [path]) for path in members]
    else:
        variable_maps = {
            variable: {parse_member(path): path for path in (root / variable).glob(f"*.{variable}.nc")}
            for variable in ("U", "V", "T")
        }
        common = sorted(set.intersection(*(set(mapping) for mapping in variable_maps.values())))
        grouped = [(member, [variable_maps[v][member] for v in ("U", "V", "T")]) for member in common]
    if len(grouped) != 30:
        raise RuntimeError(f"{case}: expected 30 complete U/V/T members; found {len(grouped)}")
    canonical_ep = None
    if output_kind == "dynamics":
        with xr.open_dataset(product_path("epflux", f"hindcast_{case}_epflux.nc"), decode_times=False) as source:
            canonical_ep = source.load()
    output_arrays, labels, common_date = [], [], None
    for label_source, paths in grouped:
        if pressure_level:
            u, v, t, dates = pressure_uvt(paths, EP_RESTART_PLEV_HPA)
            label = parse_member(label_source) if isinstance(label_source, Path) else str(label_source)
        else:
            # Each CAM member has separate U/V/T files; interpolate with PS from U.
            with xr.open_dataset(paths[0], decode_times=True, chunks={"time": 8}) as du, \
                 xr.open_dataset(paths[1], decode_times=True, chunks={"time": 8}) as dv, \
                 xr.open_dataset(paths[2], decode_times=True, chunks={"time": 8}) as dt:
                dates = date_int(du)
                for variable, source in (("V", dv), ("T", dt)):
                    if not np.array_equal(dates, date_int(source)):
                        raise RuntimeError(f"{case} {label_source}: U/{variable} dates differ")
                    for coordinate in ("time", "lat", "lon"):
                        if not np.array_equal(np.asarray(du["U"][coordinate].values),
                                              np.asarray(source[variable][coordinate].values)):
                            raise RuntimeError(
                                f"{case} {label_source}: U/{variable} {coordinate} differs"
                            )
                pressure = hybrid_mid_pressure(du)
                u = log_pressure_interpolate(du["U"], pressure, PLEV_HPA).load()
                v = log_pressure_interpolate(dv["V"], pressure, PLEV_HPA).load()
                t = log_pressure_interpolate(dt["T"], pressure, PLEV_HPA).load()
            label = str(label_source)
        ep = (
            compute_epflux_monthly(u, v, t, dates, ComputeEPfluxDiv)
            if output_kind == "epflux" else canonical_ep.sel(member=label, drop=True)
        )
        dynamics = dynamics_from_uvt(u, t, ep)
        ep_dates = np.asarray(ep.date.values, dtype=np.int32)
        if common_date is None:
            common_date = ep_dates
        elif not np.array_equal(common_date, ep_dates):
            raise RuntimeError(f"{case}: member calendars differ")
        output_arrays.append(
            ep.drop_vars("date") if output_kind == "epflux" else dynamics.drop_vars("date")
        )
        labels.append(label)
    coordinate = xr.IndexVariable("member", labels)
    output = xr.concat(output_arrays, dim=coordinate).assign_coords(date=("time", common_date))
    output.attrs.update(
        product_version=PRODUCT_VERSION, case=case, member_count=30,
        natural_month_n2="True", do_ubar="True", w_argument="None", wave="-1",
    )
    if output_kind == "epflux":
        write_ep(
            output, f"hindcast_{case}_epflux.nc",
            expected_plev=EP_RESTART_PLEV_HPA, members=True,
        )
    else:
        write_dynamics(output, f"hindcast_{case}.nc", members=True)


## January, February, and March restart EP flux

Inputs: 30-member U/V/T cases. Outputs: epflux/hindcast_{case}_epflux.nc. Method: natural-month N2, do_ubar=True, w=None, wave=-1, full-latitude upward flux and 40--80N EP100.


In [ ]:
hindcast_case("0008-01", PREPROCESSED_ROOT / "Hindcast" / "0008-01", pressure_level=True, output_kind="epflux")
hindcast_case("0008-02", PREPROCESSED_ROOT / "Hindcast" / "0008-02", pressure_level=True, output_kind="epflux")
hindcast_case("0008-03", MARCH_HINDCAST_ROOT, pressure_level=True, output_kind="epflux")


## January, February, and March U60N10/Tmin50 dynamics

Inputs: the same 30-member U/T cases and identically defined EP100. Outputs: dynamics/hindcast_{case}.nc. Method: compute only U60N10, Tmin50, and EP100 for selected figures.


In [ ]:
hindcast_case("0008-01", PREPROCESSED_ROOT / "Hindcast" / "0008-01", pressure_level=True, output_kind="dynamics")
hindcast_case("0008-02", PREPROCESSED_ROOT / "Hindcast" / "0008-02", pressure_level=True, output_kind="dynamics")
hindcast_case("0008-03", MARCH_HINDCAST_ROOT, pressure_level=True, output_kind="dynamics")
